# Regresión: predecir un número

**Sección 3.14 del libro**

Ahora no queremos una categoría sino un valor continuo: el **largo del sépalo** a partir de las otras medidas.

> **Basado en:** Karatas, M. (2024). *Developing AI Applications: An Introduction*, cap. 3 «Artificial Neural Networks». Rheinwerk Publishing. Código original del autor, adaptado a Google Colab y comentado en español con fines educativos.


In [ ]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Cargar los datos

Antes de ejecutar esta celda, sube **iris.csv** al panel de archivos de Colab (ícono de carpeta → botón de subir). Si no lo tienes a mano, usa la línea comentada: lee el archivo directamente desde la página de la presentación.

In [ ]:
path = "iris.csv"
# path = "https://sinergia-iesa.github.io/introduccion-a-la-creacion-de-redes-neuronales/data/iris.csv"   # alternativa sin subir el archivo

data = pd.read_csv(path, delimiter=',')
print(data.head())
print("Columnas con huecos:", data.columns[data.isnull().any()])

In [ ]:
# Correlaciones: ¿qué columnas aportan poco?
correlations = data.corr(numeric_only=True)
correlations_abs_sum = correlations.abs().sum()
print('Suma de correlaciones (en valor absoluto)')
print('-' * 30)
print(correlations_abs_sum)
print('\nLas 5 más débiles')
print('-' * 30)
print(correlations_abs_sum.nsmallest(5))

`sepal.width` tiene la correlación más baja. La quitamos (¡hay que probar si ayuda!).

In [ ]:
data = data.drop(['sepal.width'], axis=1)

col = data['sepal.length']                  # lo que vamos a predecir
data = data.drop(['sepal.length'], axis=1)

data = pd.get_dummies(data, columns=['species'], dtype=float)

s_scaler = StandardScaler()
cols_to_s_scale = ['petal.length', 'petal.width']
data[cols_to_s_scale] = s_scaler.fit_transform(data[cols_to_s_scale])

train_data, test_data, train_col, test_col = train_test_split(data, col, test_size=0.2, random_state=42)

## La red de regresión

La salida es **1 neurona sin activación**: entrega el número tal cual. La pérdida es **MAE** (error absoluto medio), en las mismas unidades que el dato (cm).

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(data.shape[1],)),
    tf.keras.layers.Dense(32, activation='sigmoid'),
    tf.keras.layers.Dense(64, activation='sigmoid'),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer='adam', loss='mae', metrics=['mae'])
model.fit(train_data, train_col, epochs=200, verbose=0)

test_loss, test_mae = model.evaluate(test_data, test_col)
print('Error medio (cm):', test_mae)

El libro llega a unos 0,3 cm de error. Sube o baja las épocas y compara.